<a href="https://colab.research.google.com/github/AbhijithDuggaraju/ProjectClubConnect/blob/main/ProjectClub.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
College Club Registration System
---------------------------------
Demonstrates: Linked List (clubs), Queue (member approvals),
              Stack (event registrations), SQLite persistence
"""

import sqlite3
import getpass
import re
import hashlib
import datetime
from collections import deque



MANAGEMENT_USERNAME = "GOABROS"
MANAGEMENT_PASSWORD_HASH = hashlib.sha256(b"RAJ").hexdigest()

EMAIL_REGEX = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'


def hash_password(password: str) -> str:
    return hashlib.sha256(password.encode()).hexdigest()


def is_valid_email(email: str) -> bool:
    return re.match(EMAIL_REGEX, email) is not None


def today() -> str:
    """Return today's date as YYYY-MM-DD string."""
    return datetime.date.today().isoformat()


def safe_int_input(prompt: str) -> int | None:
    """Prompt for an integer; return None if the input is not a valid integer."""
    raw = input(prompt).strip()
    try:
        return int(raw)
    except ValueError:
        print(f"  [!] Expected a number, got '{raw}'. Please try again.")
        return None




class MemberQueue:
    """FIFO queue for pending member registration requests."""

    def __init__(self):
        self._queue = deque()

    def enqueue(self, name: str, email: str, club_id: int, registration_date: str):
        self._queue.append((name, email, club_id, registration_date))

    def dequeue(self):
        return self._queue.popleft() if self._queue else None

    def is_empty(self) -> bool:
        return len(self._queue) == 0

    def display_pending(self):
        if self.is_empty():
            print("  No pending registrations.")
            return
        print("\n  Pending Member Registrations:")
        print("  " + "-" * 55)
        for idx, (name, email, club_id, reg_date) in enumerate(self._queue, 1):
            print(f"  {idx}. {name:<20} {email:<25} Club ID: {club_id}  Date: {reg_date}")
        print("  " + "-" * 55)


class ClubNode:
    """Node in the Club linked list."""

    def __init__(self, club_id: int, club_name: str, description: str, faculty: str):
        self.club_id = club_id
        self.club_name = club_name
        self.description = description
        self.faculty = faculty
        self.next: "ClubNode | None" = None


class ClubLinkedList:
    """Singly linked list storing all approved clubs."""

    def __init__(self):
        self.head: ClubNode | None = None

    def add_club(self, club_id: int, club_name: str, description: str, faculty: str):
        node = ClubNode(club_id, club_name, description, faculty)
        node.next = self.head
        self.head = node

    def get_club(self, club_id: int) -> ClubNode | None:
        current = self.head
        while current:
            if current.club_id == club_id:
                return current
            current = current.next
        return None

    def display_clubs(self):
        current = self.head
        if not current:
            print("  No clubs available.")
            return
        print("\n  Available Clubs:")
        print("  " + "-" * 70)
        while current:
            print(f"  ID: {current.club_id:<4} {current.club_name:<20} Faculty: {current.faculty}")
            print(f"       {current.description}")
            current = current.next
        print("  " + "-" * 70)


class EventStack:
    """LIFO stack for tracking the most-recent event registrations this session."""

    def __init__(self):
        self._stack: list = []

    def push(self, event_id: int, student_id: int, registration_date: str):
        self._stack.append((event_id, student_id, registration_date))

    def pop(self):
        return self._stack.pop() if self._stack else None

    def is_empty(self) -> bool:
        return len(self._stack) == 0

    def peek(self):
        return self._stack[-1] if self._stack else None



club_linked_list: ClubLinkedList = ClubLinkedList()
member_queue: MemberQueue = MemberQueue()
event_stack: EventStack = EventStack()
event_registration_queue: deque = deque()



DB_PATH = "club_registration.db"


def connect_db() -> sqlite3.Connection:
    conn = sqlite3.connect(DB_PATH)
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


def create_tables():
    conn = connect_db()
    cur = conn.cursor()

    cur.executescript("""
        CREATE TABLE IF NOT EXISTS clubs (
            club_id           INTEGER PRIMARY KEY AUTOINCREMENT,
            club_name         TEXT NOT NULL,
            description       TEXT NOT NULL,
            faculty           TEXT NOT NULL
        );

        CREATE TABLE IF NOT EXISTS members (
            member_id         INTEGER PRIMARY KEY AUTOINCREMENT,
            name              TEXT NOT NULL,
            email             TEXT NOT NULL UNIQUE,
            password_hash     TEXT NOT NULL,
            registration_date TEXT NOT NULL,
            club_id           INTEGER REFERENCES clubs(club_id)
        );

        CREATE TABLE IF NOT EXISTS feedback (
            feedback_id  INTEGER PRIMARY KEY AUTOINCREMENT,
            club_id      INTEGER REFERENCES clubs(club_id),
            student_name TEXT NOT NULL,
            feedback     TEXT NOT NULL,
            submitted_at TEXT NOT NULL
        );

        CREATE TABLE IF NOT EXISTS pending_clubs (
            request_id   INTEGER PRIMARY KEY AUTOINCREMENT,
            club_name    TEXT NOT NULL,
            description  TEXT NOT NULL,
            faculty      TEXT NOT NULL,
            requested_at TEXT NOT NULL
        );

        CREATE TABLE IF NOT EXISTS events (
            event_id    INTEGER PRIMARY KEY AUTOINCREMENT,
            club_id     INTEGER REFERENCES clubs(club_id),
            event_name  TEXT NOT NULL,
            event_date  TEXT NOT NULL,
            description TEXT NOT NULL
        );

        CREATE TABLE IF NOT EXISTS event_registrations (
            registration_id   INTEGER PRIMARY KEY AUTOINCREMENT,
            event_id          INTEGER REFERENCES events(event_id),
            student_id        INTEGER REFERENCES members(member_id),
            registration_date TEXT NOT NULL
        );
    """)


    cur.execute("SELECT COUNT(*) FROM clubs")
    if cur.fetchone()[0] == 0:
        default_clubs = [
            ("Art Club",        "A club for students interested in visual arts.",         "Ms. Devi"),
            ("Science Club",    "Exploring the wonders of science.",                      "Dr. Naidu"),
            ("Literature Club", "Discussing and analysing great works of literature.",    "Mr. Gangadhar"),
            ("Coding Club",     "For those interested in coding and software development.","Dr. Jayaram"),
            ("Music Club",      "A club for music lovers and musicians.",                 "Ms. Fatima"),
        ]
        cur.executemany(
            "INSERT INTO clubs (club_name, description, faculty) VALUES (?, ?, ?)",
            default_clubs,
        )

    conn.commit()
    conn.close()


def load_clubs_into_linked_list():
    global club_linked_list
    club_linked_list = ClubLinkedList()
    conn = connect_db()
    cur = conn.cursor()
    cur.execute("SELECT club_id, club_name, description, faculty FROM clubs")
    for row in cur.fetchall():
        club_linked_list.add_club(*row)
    conn.close()



def student_login():
    print("\n--- Student Login ---")
    print("  (Enter your registered email and password.)")
    email = input("  Email: ").strip()
    if not is_valid_email(email):
        print("  [!] Invalid email format.")
        return

    password = getpass.getpass("  Password: ")
    pw_hash = hash_password(password)

    conn = connect_db()
    cur = conn.cursor()
    cur.execute(
        "SELECT member_id, name FROM members WHERE email = ? AND password_hash = ?",
        (email, pw_hash),
    )
    row = cur.fetchone()
    conn.close()

    if row:
        print(f"  Login successful. Welcome, {row[1]}!")
        student_menu()
    else:
        print("  [!] Invalid credentials. Access denied.")


def management_login():
    print("\n--- Management Login ---")
    username = input("  Username: ").strip()
    password = getpass.getpass("  Password: ")

    if username == MANAGEMENT_USERNAME and hash_password(password) == MANAGEMENT_PASSWORD_HASH:
        print("  Login successful.")
        management_menu()
    else:
        print("  [!] Invalid credentials. Access denied.")



def main_menu():
    while True:
        print("\n" + "=" * 40)
        print("  College Club Registration System")
        print("=" * 40)
        print("  1. Management Login")
        print("  2. Student Login")
        print("  3. Register as New Student")
        print("  4. Exit")

        choice = input("  Choice (1-4): ").strip()
        if choice == "1":
            management_login()
        elif choice == "2":
            student_login()
        elif choice == "3":
            register_new_student()
        elif choice == "4":
            print("  Goodbye!")
            break
        else:
            print("  [!] Invalid choice.")


def student_menu():
    options = {
        "1": ("Display clubs",           display_club_list),
        "2": ("View club details",        display_club_details),
        "3": ("Register for a club",      register_new_member),
        "4": ("Submit feedback",          submit_feedback),
        "5": ("Propose a new club",       register_new_club),
        "6": ("Register for an event",    register_for_event),
        "7": ("Back to main menu",        None),
    }
    _run_menu("Student Menu", options)


def management_menu():
    options = {
        "1": ("Display clubs",                display_club_list),
        "2": ("Approve new club proposals",   approve_new_clubs),
        "3": ("Approve member registrations", approve_member_registrations),
        "4": ("Update club details",          update_club_details),
        "5": ("Add event to a club",          add_event_to_club),
        "6": ("View feedback",                view_all_feedback),
        "7": ("Back to main menu",            None),
    }
    _run_menu("Management Menu", options)


def _run_menu(title: str, options: dict):
    while True:
        print(f"\n--- {title} ---")
        for key, (label, _) in options.items():
            print(f"  {key}. {label}")
        choice = input(f"  Choice (1-{len(options)}): ").strip()
        if choice not in options:
            print("  [!] Invalid choice.")
            continue
        label, fn = options[choice]
        if fn is None:
            print("  Returning to previous menu.")
            break
        fn()



def register_new_student():
    print("\n--- Create Student Account ---")
    name = input("  Full name: ").strip()
    if not name:
        print("  [!] Name cannot be empty.")
        return

    email = input("  Email: ").strip()
    if not is_valid_email(email):
        print("  [!] Invalid email format.")
        return

    password = getpass.getpass("  Choose a password: ")
    confirm  = getpass.getpass("  Confirm password: ")
    if password != confirm:
        print("  [!] Passwords do not match.")
        return
    if len(password) < 6:
        print("  [!] Password must be at least 6 characters.")
        return

    pw_hash = hash_password(password)

    try:
        conn = connect_db()
        cur = conn.cursor()
        cur.execute(
            "INSERT INTO members (name, email, password_hash, registration_date) VALUES (?, ?, ?, ?)",
            (name, email, pw_hash, today()),
        )
        conn.commit()
        conn.close()
        print(f"  Account created for {name}. You can now log in.")
    except sqlite3.IntegrityError:
        print("  [!] An account with that email already exists.")



def display_club_list():
    club_linked_list.display_clubs()


def display_club_details():
    display_club_list()
    club_id = safe_int_input("  Enter club ID: ")
    if club_id is None:
        return

    club = club_linked_list.get_club(club_id)
    if not club:
        print("  [!] Club not found.")
        return

    print(f"\n  Club: {club.club_name}")
    print(f"  Description : {club.description}")
    print(f"  Faculty     : {club.faculty}")

    conn = connect_db()
    cur = conn.cursor()

    cur.execute("SELECT name FROM members WHERE club_id = ?", (club_id,))
    members = cur.fetchall()
    print(f"\n  Members ({len(members)}):")
    if members:
        for (m,) in members:
            print(f"    - {m}")
    else:
        print("    No members yet.")

    cur.execute(
        "SELECT student_name, feedback, submitted_at FROM feedback WHERE club_id = ?",
        (club_id,),
    )
    entries = cur.fetchall()
    print(f"\n  Feedback ({len(entries)}):")
    if entries:
        for student_name, fb, submitted_at in entries:
            print(f"    [{submitted_at}] {student_name}: {fb}")
    else:
        print("    No feedback yet.")

    conn.close()


def register_new_member():
    """Add a member registration request to the approval queue."""
    display_club_list()
    club_id = safe_int_input("  Enter club ID to join: ")
    if club_id is None:
        return
    if not club_linked_list.get_club(club_id):
        print("  [!] Club not found.")
        return

    name  = input("  Your name: ").strip()
    email = input("  Your email: ").strip()
    if not is_valid_email(email):
        print("  [!] Invalid email format.")
        return

    member_queue.enqueue(name, email, club_id, today())
    print(f"  Registration request submitted for {name}. Awaiting management approval.")


def update_club_details():
    display_club_list()
    club_id = safe_int_input("  Enter club ID to update: ")
    if club_id is None:
        return

    club = club_linked_list.get_club(club_id)
    if not club:
        print("  [!] Club not found.")
        return

    print(f"  Current description : {club.description}")
    print(f"  Current faculty     : {club.faculty}")

    new_desc    = input("  New description (Enter to keep current): ").strip()
    new_faculty = input("  New faculty name (Enter to keep current): ").strip()

    if new_desc:
        club.description = new_desc
    if new_faculty:
        club.faculty = new_faculty

    conn = connect_db()
    conn.execute(
        "UPDATE clubs SET description = ?, faculty = ? WHERE club_id = ?",
        (club.description, club.faculty, club_id),
    )
    conn.commit()
    conn.close()
    print("  Club details updated.")


def register_new_club():
    print("\n--- Propose a New Club ---")
    club_name   = input("  Club name: ").strip()
    description = input("  Description: ").strip()
    faculty     = input("  Faculty advisor: ").strip()

    if not club_name or not description or not faculty:
        print("  [!] All fields are required.")
        return

    conn = connect_db()
    conn.execute(
        "INSERT INTO pending_clubs (club_name, description, faculty, requested_at) VALUES (?, ?, ?, ?)",
        (club_name, description, faculty, today()),
    )
    conn.commit()
    conn.close()
    print(f"  Club '{club_name}' proposed successfully. Awaiting management approval.")



def submit_feedback():
    display_club_list()
    club_id = safe_int_input("  Enter club ID: ")
    if club_id is None:
        return
    if not club_linked_list.get_club(club_id):
        print("  [!] Club not found.")
        return

    student_name  = input("  Your name: ").strip()
    feedback_text = input("  Feedback: ").strip()
    if not feedback_text:
        print("  [!] Feedback cannot be empty.")
        return

    conn = connect_db()
    conn.execute(
        "INSERT INTO feedback (club_id, student_name, feedback, submitted_at) VALUES (?, ?, ?, ?)",
        (club_id, student_name, feedback_text, today()),
    )
    conn.commit()
    conn.close()
    print("  Feedback submitted. Thank you!")


def view_all_feedback():
    conn = connect_db()
    cur = conn.cursor()
    cur.execute("""
        SELECT c.club_name, f.student_name, f.feedback, f.submitted_at
        FROM feedback f
        JOIN clubs c ON c.club_id = f.club_id
        ORDER BY f.submitted_at DESC
    """)
    rows = cur.fetchall()
    conn.close()

    if not rows:
        print("  No feedback submitted yet.")
        return

    print("\n  All Feedback:")
    print("  " + "-" * 70)
    for club_name, student, fb, submitted_at in rows:
        print(f"  [{submitted_at}] {club_name} — {student}")
        print(f"    {fb}")
    print("  " + "-" * 70)



def approve_new_clubs():
    conn = connect_db()
    cur = conn.cursor()
    cur.execute("SELECT request_id, club_name, description, faculty FROM pending_clubs")
    pending = cur.fetchall()

    if not pending:
        print("  No pending club proposals.")
        conn.close()
        return

    print("\n  Pending Club Proposals:")
    print("  " + "-" * 60)
    for req_id, name, desc, faculty in pending:
        print(f"  [{req_id}] {name:<20} Faculty: {faculty}")
        print(f"       {desc}")
    print("  " + "-" * 60)

    request_id = safe_int_input("  Enter Request ID to approve (0 to cancel): ")
    if request_id is None or request_id == 0:
        conn.close()
        return

    cur.execute(
        "SELECT club_name, description, faculty FROM pending_clubs WHERE request_id = ?",
        (request_id,),
    )
    club_data = cur.fetchone()
    if not club_data:
        print("  [!] Invalid Request ID.")
        conn.close()
        return

    club_name, description, faculty = club_data

    cur.execute(
        "INSERT INTO clubs (club_name, description, faculty) VALUES (?, ?, ?)",
        (club_name, description, faculty),
    )
    new_club_id = cur.lastrowid

    cur.execute("DELETE FROM pending_clubs WHERE request_id = ?", (request_id,))
    conn.commit()
    conn.close()


    club_linked_list.add_club(new_club_id, club_name, description, faculty)
    print(f"  Club '{club_name}' approved and added (ID {new_club_id}).")


def approve_member_registrations():
    member_queue.display_pending()
    if member_queue.is_empty():
        return

    choice = safe_int_input("  Enter position number to approve (0 to cancel): ")
    if choice is None or choice == 0:
        return


    pending_member = member_queue.dequeue()
    if not pending_member:
        print("  [!] Queue is empty.")
        return

    name, email, club_id, reg_date = pending_member


    if not club_linked_list.get_club(club_id):
        print(f"  [!] Club ID {club_id} no longer exists. Registration discarded.")
        return

    try:
        conn = connect_db()

        cur = conn.cursor()
        cur.execute("SELECT member_id FROM members WHERE email = ?", (email,))
        row = cur.fetchone()

        if row:
            member_id = row[0]
            cur.execute("UPDATE members SET club_id = ? WHERE member_id = ?", (club_id, member_id))
        else:

            temp_hash = hash_password(email)
            cur.execute(
                "INSERT INTO members (name, email, password_hash, registration_date, club_id) VALUES (?, ?, ?, ?, ?)",
                (name, email, temp_hash, reg_date, club_id),
            )
            print(f"  Note: Temporary account created. Initial password is the registered email address.")

        conn.commit()
        conn.close()
        print(f"  Approved: {name} is now a member of Club ID {club_id}.")
    except sqlite3.IntegrityError as e:
        print(f"  [!] Database error: {e}")



def add_event_to_club():
    display_club_list()
    club_id = safe_int_input("  Enter club ID: ")
    if club_id is None:
        return
    if not club_linked_list.get_club(club_id):
        print("  [!] Club not found.")
        return

    event_name   = input("  Event name: ").strip()
    event_date   = input("  Event date (YYYY-MM-DD): ").strip()
    event_desc   = input("  Description: ").strip()

    if not event_name or not event_date or not event_desc:
        print("  [!] All fields are required.")
        return


    try:
        datetime.date.fromisoformat(event_date)
    except ValueError:
        print("  [!] Invalid date format. Use YYYY-MM-DD.")
        return

    conn = connect_db()
    conn.execute(
        "INSERT INTO events (club_id, event_name, event_date, description) VALUES (?, ?, ?, ?)",
        (club_id, event_name, event_date, event_desc),
    )
    conn.commit()
    conn.close()
    print(f"  Event '{event_name}' added to club ID {club_id}.")


def register_for_event():
    display_club_list()
    club_id = safe_int_input("  Enter club ID to browse events: ")
    if club_id is None:
        return

    conn = connect_db()
    cur = conn.cursor()
    cur.execute(
        "SELECT event_id, event_name, event_date, description FROM events WHERE club_id = ?",
        (club_id,),
    )
    events = cur.fetchall()

    if not events:
        print("  No events for this club.")
        conn.close()
        return

    print("\n  Upcoming Events:")
    print("  " + "-" * 60)
    for ev_id, ev_name, ev_date, ev_desc in events:
        print(f"  ID: {ev_id}  {ev_name}  [{ev_date}]")
        print(f"       {ev_desc}")
    print("  " + "-" * 60)

    event_id = safe_int_input("  Enter event ID to register for: ")
    if event_id is None:
        conn.close()
        return


    cur.execute("SELECT event_name FROM events WHERE event_id = ?", (event_id,))
    ev_row = cur.fetchone()
    if not ev_row:
        print("  [!] Event not found.")
        conn.close()
        return

    student_id = safe_int_input("  Enter your student/member ID: ")
    if student_id is None:
        conn.close()
        return


    cur.execute(
        "SELECT 1 FROM event_registrations WHERE event_id = ? AND student_id = ?",
        (event_id, student_id),
    )
    if cur.fetchone():
        print("  [!] You are already registered for this event.")
        conn.close()
        return

    reg_date = today()
    cur.execute(
        "INSERT INTO event_registrations (event_id, student_id, registration_date) VALUES (?, ?, ?)",
        (event_id, student_id, reg_date),
    )
    conn.commit()


    event_stack.push(event_id, student_id, reg_date)

    event_registration_queue.append((event_id, student_id, reg_date))

    print(f"  Registered for '{ev_row[0]}' on {reg_date}.")
    _display_event_participants(cur, event_id)
    conn.close()


def _display_event_participants(cur, event_id: int):
    cur.execute(
        """SELECT m.name, er.registration_date
           FROM event_registrations er
           JOIN members m ON m.member_id = er.student_id
           WHERE er.event_id = ?
           ORDER BY er.registration_date""",
        (event_id,),
    )
    rows = cur.fetchall()
    print(f"\n  Registered participants ({len(rows)}):")
    for name, reg_date in rows:
        print(f"    - {name}  [{reg_date}]")


if __name__ == "__main__":
    create_tables()
    load_clubs_into_linked_list()
    main_menu()


  College Club Registration System
  1. Management Login
  2. Student Login
  3. Register as New Student
  4. Exit
  Choice (1-4): 2

--- Student Login ---
  (Enter your registered email and password.)
  Email: duggaraju53@gmail.com
  Password: ··········
  [!] Invalid credentials. Access denied.

  College Club Registration System
  1. Management Login
  2. Student Login
  3. Register as New Student
  4. Exit
  Choice (1-4): 4
  Goodbye!
